# LEMON Full Batch Preprocessing Only

This notebook is a clean, batch-only version of preprocessing.
It avoids the single-subject demo flow and runs directly on all subjects.

## Sub-steps
1. Imports and environment checks
2. Paths and parameters
3. Helper functions (bad channels, bad segments)
4. Subject-level pipeline function
5. Batch loop (all subjects)
6. Summary and CSV export

In [1]:
# Sub-step 1: Imports and environment checks
import os
import glob
import time
import warnings

import mne
import numpy as np
import pandas as pd
from mne_icalabel import label_components

wandb = None
WANDB_AVAILABLE = False
try:
    import wandb as wandb_module
    wandb = wandb_module
    WANDB_AVAILABLE = True
except ImportError:
    pass

print(f'MNE version: {mne.__version__}')
mne.set_log_level('WARNING')
RANDOM_STATE = 42
MNE_CUDA = False
try:
    mne.utils.set_config('MNE_USE_CUDA', 'false', set_env=True)
except Exception:
    pass
print('Forcing CPU mode: MNE_CUDA=False')

MNE version: 1.11.0
Forcing CPU mode: MNE_CUDA=False


In [ ]:
#parameters
LEMON_DIR = r'G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\EEG_MPILMBB_LEMON\\EEG_Raw_BIDS_ID'
OUTPUT_DIR = r'G:\\Study\\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\\data\\LEMON_preprocessed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SFREQ = 250
LINE_NOISE_FREQ = 50
HIGHPASS_FREQ = 1.0
LOWPASS_FREQ = 100.0
IC_REJECTION_THRESHOLD = 0.80
EPOCH_DURATION = 4.0
EPOCH_OVERLAP = 0.5
AMPLITUDE_REJECT_UV = 250e-6

# QC thresholds (configurable)
IC_REJECT_FRACTION_FLAG = 0.25  # flag if >=25% of ICA components rejected
EPOCH_KEEP_RATIO_FLAG = 0.5     # flag if kept epochs <=50% of before

subject_dirs = sorted(glob.glob(os.path.join(LEMON_DIR, 'sub-*')))
print(f'Found {len(subject_dirs)} subjects')
print(f'Output dir: {OUTPUT_DIR}')

Found 220 subjects
Output dir: G:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\data\LEMON_preprocessed


In [6]:
# Sub-step 2b: Weights & Biases tracking setup
WANDB_ENTITY = 'amahbe223043-united-international-university'
WANDB_PROJECT = 'LEMON_DATASET_preprocessing'
WANDB_RUN_NAME = f'lemon-preprocessing-{time.strftime("%Y%m%d-%H%M%S")}'

# Increase init timeout to avoid startup failures on slow/unreliable networks.
WANDB_INIT_TIMEOUT = int(os.getenv('WANDB_INIT_TIMEOUT', '180'))

wandb_run = None
wandb_table = None
wandb_subject_rows = []

if WANDB_AVAILABLE:
    try:
        wandb_api_key = os.getenv('WANDB_API_KEY')

        # If key is present in env, use it explicitly; otherwise rely on existing `wandb login` session.
        if wandb_api_key:
            wandb.login(key=wandb_api_key)

        wandb_run = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name=WANDB_RUN_NAME,
            config={
                'target_sfreq': TARGET_SFREQ,
                'line_noise_freq': LINE_NOISE_FREQ,
                'highpass_freq': HIGHPASS_FREQ,
                'lowpass_freq': LOWPASS_FREQ,
                'ic_rejection_threshold': IC_REJECTION_THRESHOLD,
                'epoch_duration': EPOCH_DURATION,
                'epoch_overlap': EPOCH_OVERLAP,
                'amplitude_reject_uv': AMPLITUDE_REJECT_UV,
                'random_state': RANDOM_STATE,
                'use_cuda': MNE_CUDA,
                'total_subjects': len(subject_dirs),
            },
            reinit='finish_previous',
            settings=wandb.Settings(init_timeout=WANDB_INIT_TIMEOUT),
        )
        wandb.define_metric('preprocessing/subject_index')
        wandb.define_metric('preprocessing/*', step_metric='preprocessing/subject_index')
        print(f'W&B tracking enabled: {WANDB_ENTITY}/{WANDB_PROJECT}')
        print(f'W&B run URL: {wandb_run.get_url()}')
    except Exception as exc:
        print(f'W&B setup failed ({type(exc).__name__}: {exc}). Retrying in offline mode...')
        try:
            wandb_run = wandb.init(
                entity=WANDB_ENTITY,
                project=WANDB_PROJECT,
                name=WANDB_RUN_NAME,
                mode='offline',
                config={
                    'target_sfreq': TARGET_SFREQ,
                    'line_noise_freq': LINE_NOISE_FREQ,
                    'highpass_freq': HIGHPASS_FREQ,
                    'lowpass_freq': LOWPASS_FREQ,
                    'ic_rejection_threshold': IC_REJECTION_THRESHOLD,
                    'epoch_duration': EPOCH_DURATION,
                    'epoch_overlap': EPOCH_OVERLAP,
                    'amplitude_reject_uv': AMPLITUDE_REJECT_UV,
                    'random_state': RANDOM_STATE,
                    'use_cuda': MNE_CUDA,
                    'total_subjects': len(subject_dirs),
                },
                settings=wandb.Settings(init_timeout=WANDB_INIT_TIMEOUT),
            )
            wandb.define_metric('preprocessing/subject_index')
            wandb.define_metric('preprocessing/*', step_metric='preprocessing/subject_index')
            print('W&B offline mode enabled. Run `wandb sync` later to upload logs.')
        except Exception as offline_exc:
            print(f'W&B offline fallback failed ({type(offline_exc).__name__}: {offline_exc}). Continuing without W&B logging.')
            wandb_run = None
else:
    print('W&B is not installed. Add wandb to requirements.txt or install it to enable tracking.')

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


W&B tracking enabled: amahbe223043-united-international-university/LEMON_DATASET_preprocessing
W&B run URL: https://wandb.ai/amahbe223043-united-international-university/LEMON_DATASET_preprocessing/runs/p3hl9f37


In [7]:
#bad-channel helper (PyPREP + RANSAC fallback)
def detect_bad_channels(raw, random_state=42, include_ransac=True):
    try:
        from pyprep.find_noisy_channels import NoisyChannels
    except ImportError as exc:
        raise ImportError('PyPREP is required. Install with: pip install pyprep') from exc

    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False, exclude=[])
    eeg_names_all = [raw.ch_names[p] for p in picks_eeg]
    eeg_names = [ch for ch in eeg_names_all if ch != 'FCz']
    raw_prep = raw.copy().pick_channels(eeg_names, ordered=True)

    noisy = NoisyChannels(raw_prep, random_state=random_state)
    noisy.find_bad_by_nan_flat()
    noisy.find_bad_by_deviation()
    noisy.find_bad_by_hfnoise()
    noisy.find_bad_by_correlation()

    ransac_used = False
    if include_ransac:
        try:
            noisy.find_bad_by_ransac()
            ransac_used = True
        except Exception as exc:
            warnings.warn(
                f'PyPREP RANSAC failed ({type(exc).__name__}: {exc}). Continuing without RANSAC.',
                RuntimeWarning
            )

    bad_channels = sorted(set(noisy.get_bads()))
    return bad_channels, ransac_used

In [8]:
#bad-time-segment helper
def annotate_bad_segments(raw, threshold_uv=250, window_s=0.5):
    sfreq = raw.info['sfreq']
    window_samples = int(window_s * sfreq)
    picks_eeg = mne.pick_types(raw.info, eeg=True, eog=False) # pick all EEG channels except EOG 
    data = raw.get_data(picks=picks_eeg) * 1e6
    n_windows = data.shape[1] // window_samples
    bad_annotations = []
    annotation_orig_time = raw.annotations.orig_time

    for w in range(n_windows):
        start = w * window_samples
        end = start + window_samples
        window_data = data[:, start:end]
        ptp = np.ptp(window_data, axis=1) #compute peak to peak amplitude for each channel in the window
        if np.any(ptp > threshold_uv):
            bad_annotations.append((start / sfreq, window_s, 'BAD_artifact'))

    if bad_annotations:
        onsets = [a[0] for a in bad_annotations] #extract onsets from thhe annotations list
        durations = [a[1] for a in bad_annotations] #extract durations from the annotations list
        desc = [a[2] for a in bad_annotations] #extract descriptions from the annotations list
        bad_annot = mne.Annotations(onset=onsets, duration=durations, description=desc, orig_time=annotation_orig_time)
        raw.set_annotations(raw.annotations + bad_annot) #add the new annotations to the raw object

    return len(bad_annotations) * window_s #return the total duration of bad segments in seconds 

In [ ]:
#subject-level preprocessing pipeline
def preprocess_lemon_subject(vhdr_file, output_dir,#vhdr is the brainvision header file path
                           target_sfreq=250,
                           line_noise_freq=50,
                           highpass_freq=1.0,
                           lowpass_freq=100.0,
                           ic_threshold=0.80,
                           epoch_duration=4.0,
                           epoch_overlap_frac=0.5,
                           amplitude_reject_uv=250e-6,
                           random_state=42,
                           use_cuda=False):
    n_jobs = 'cuda' if use_cuda else 1
    subject_id = os.path.basename(os.path.dirname(os.path.dirname(vhdr_file)))
    result = {'subject': subject_id, 'status': 'success'}

    try:
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        # Channel setup
        raw.set_channel_types({'VEOG': 'eog'})
        if 'FCz' not in raw.ch_names:
            raw = mne.add_reference_channels(raw, ref_channels=['FCz'])
        raw.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)

        #Filtering + resampling
        raw.notch_filter(freqs=[line_noise_freq, line_noise_freq * 2], method='spectrum_fit', verbose=False)
        #this removes powerline noise and its first harmonic,which are common contaminans in EEG data..
        raw.filter(l_freq=highpass_freq, h_freq=lowpass_freq, n_jobs=n_jobs, verbose=False)
        #this removes slow drifts and DC offset, which can interfere with ICA and other analyses 
        raw.resample(sfreq=target_sfreq, npad='auto', n_jobs=n_jobs, verbose=False)
        #we resample to 250hz to balance temporal resolution and computational efficiency and to reduce computational time 
        #also our other dataset is at 250hz so this makes it easier to work with both dataset together

        # Bad channels
        bad_chs, ransac_used = detect_bad_channels(raw, random_state=random_state, include_ransac=True)
        raw.info['bads'] = bad_chs
        result['n_bad_channels'] = len(bad_chs)
        result['ransac_used'] = ransac_used

        dropped = list(raw.info['bads'])
        raw_clean = raw.copy().drop_channels(dropped) if dropped else raw.copy()

        # Re-reference
        raw_clean.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

        # ICA + ICLabel
        n_eeg = len(mne.pick_types(raw_clean.info, eeg=True, eog=False))
        if n_eeg < 2:
            raise RuntimeError(f'Not enough EEG channels for ICA after cleaning: {n_eeg}')
        n_components = n_eeg - 1
        ica = mne.preprocessing.ICA(
            n_components=n_components,
            method='infomax',
            fit_params=dict(extended=True),
            random_state=random_state,
            max_iter='auto',
            verbose=False
        )
        ica.fit(raw_clean, picks='eeg', verbose=False)

        ic_labels = label_components(raw_clean, ica, method='iclabel')
        labels = ic_labels.get('labels', [])
        probs = np.asarray(ic_labels.get('y_pred_proba', []))
        reject_label_set = {
            'eye blink',
            'muscle artifact',
            'heart beat',
            'line noise',
            'channel noise',
            'other',
        }
        reject_ics = []
        for i, label in enumerate(labels):
            prob_i = float(probs[i]) if probs.size > i else 1.0
            if label in reject_label_set and prob_i >= ic_threshold:
                reject_ics.append(i)
        ica.exclude = reject_ics
        raw_clean = ica.apply(raw_clean, verbose=False)
        result['n_rejected_ics'] = len(reject_ics)
        # QC: fraction of rejected ICA components
        try:
            ic_frac = len(reject_ics) / max(1, n_components)
        except Exception:
            ic_frac = 1.0 if len(reject_ics) > 0 else 0.0
        result['reject_ics_fraction'] = ic_frac
        result['flag_high_ic_reject'] = bool(ic_frac >= IC_REJECT_FRACTION_FLAG)
        if result['flag_high_ic_reject']:
            warnings.warn(f"High IC rejection fraction: {ic_frac:.1%}", RuntimeWarning)

        #interpolate dropped channels
        if dropped:
            for ch in dropped:
                if ch not in raw_clean.ch_names:
                    raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])
                    # Re-apply montage so the newly added channel has a location (silences warning)
                    raw_clean.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)
            raw_clean.info['bads'] = dropped
            raw_clean.set_montage(mne.channels.make_standard_montage('standard_1005'), on_missing='warn', verbose=False)
            raw_clean.interpolate_bads(reset_bads=True, verbose=False)

        # Bad segments + epoching
        bad_s = annotate_bad_segments(raw_clean, threshold_uv=250, window_s=0.5)
        result['bad_segments_s'] = bad_s

        raw_eeg = raw_clean.copy().pick('eeg')
        epochs = mne.make_fixed_length_epochs(
            raw_eeg,
            duration=epoch_duration,
            overlap=epoch_duration * epoch_overlap_frac,
            preload=True,
            verbose=False
        )

        n_before = len(epochs)
        epochs.drop_bad(reject=dict(eeg=amplitude_reject_uv), verbose=False)
        result['n_epochs_before'] = n_before
        result['n_epochs_after'] = len(epochs)
        result['epoch_shape'] = epochs.get_data().shape
        # QC: epoch keep ratio
        try:
            keep_ratio = result['n_epochs_after'] / max(1, result['n_epochs_before'])
        except Exception:
            keep_ratio = 0.0
        result['epoch_keep_ratio'] = keep_ratio
        result['flag_epoch_loss'] = bool(keep_ratio <= EPOCH_KEEP_RATIO_FLAG)
        if result['flag_epoch_loss']:
            warnings.warn(f"Large epoch loss: kept {keep_ratio:.1%} of epochs", RuntimeWarning)

        # Save
        fif_path = os.path.join(output_dir, f'{subject_id}-epo.fif')
        npy_path = os.path.join(output_dir, f'{subject_id}_epochs.npy')
        epochs.save(fif_path, overwrite=True, verbose=False)
        np.save(npy_path, epochs.get_data())

        ch_path = os.path.join(output_dir, 'channel_names.txt')
        if not os.path.exists(ch_path):
            with open(ch_path, 'w') as f:
                f.write('\n'.join(epochs.ch_names))

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)

    return result

In [10]:
# Sub-step 5: Run batch over all subjects
print('=' * 70)
print('LEMON DATASET - FULL BATCH PREPROCESSING')
print('=' * 70)
print(f'Subjects to process: {len(subject_dirs)}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Target sampling rate: {TARGET_SFREQ} Hz')
print(f"GPU acceleration: {'YES (CuPy)' if MNE_CUDA else 'NO'}")
print('=' * 70 + '\n')

results = []
start_time = time.time()
active_attempts = 0
success_count = 0
failed_count = 0
missing_count = 0
skipped_count = 0

for idx, subject_dir in enumerate(subject_dirs):
    subject_id = os.path.basename(subject_dir)
    vhdr_file = os.path.join(subject_dir, 'RSEEG', f'{subject_id}.vhdr')

    output_npy = os.path.join(OUTPUT_DIR, f'{subject_id}_epochs.npy')
    if os.path.exists(output_npy):
        print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - SKIPPED (already done)')
        skipped_count += 1
        results.append({'subject': subject_id, 'status': 'skipped_existing'})
        wandb_subject_rows.append({
            'subject': subject_id,
            'status': 'skipped_existing',
            'elapsed_s': 0.0,
            'n_bad_channels': 0,
            'n_rejected_ics': 0,
            'n_epochs_before': 0,
            'n_epochs_after': 0,
            'bad_segments_s': 0.0,
            'success_rate_running': success_count / max(1, active_attempts),
        })
        if wandb_run is not None:
            wandb_run.log({
                'preprocessing/subject_index': idx + 1,
                'preprocessing/subject_total': len(subject_dirs),
                'preprocessing/skipped_existing_count': skipped_count,
                'preprocessing/success_count': success_count,
                'preprocessing/failed_count': failed_count,
                'preprocessing/missing_count': missing_count,
                'preprocessing/active_attempts': active_attempts,
                'preprocessing/current_success_rate': success_count / max(1, active_attempts),
            }, step=idx + 1)
        continue

    if not os.path.exists(vhdr_file):
        print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - SKIPPED (no .vhdr file)')
        missing_count += 1
        active_attempts += 1
        result = {'subject': subject_id, 'status': 'missing_file'}
        results.append(result)
        wandb_subject_rows.append({
            'subject': subject_id,
            'status': 'missing_file',
            'elapsed_s': 0.0,
            'n_bad_channels': 0,
            'n_rejected_ics': 0,
            'n_epochs_before': 0,
            'n_epochs_after': 0,
            'bad_segments_s': 0.0,
            'success_rate_running': success_count / max(1, active_attempts),
        })
        if wandb_run is not None:
            wandb_run.log({
                'preprocessing/subject_index': idx + 1,
                'preprocessing/subject_total': len(subject_dirs),
                'preprocessing/missing_count': missing_count,
                'preprocessing/success_count': success_count,
                'preprocessing/failed_count': failed_count,
                'preprocessing/skipped_existing_count': skipped_count,
                'preprocessing/active_attempts': active_attempts,
                'preprocessing/current_success_rate': success_count / max(1, active_attempts),
            }, step=idx + 1)
        continue

    t0 = time.time()
    print(f'[{idx+1:3d}/{len(subject_dirs)}] {subject_id} - Processing...', end=' ', flush=True)

    result = preprocess_lemon_subject(
        vhdr_file=vhdr_file,
        output_dir=OUTPUT_DIR,
        target_sfreq=TARGET_SFREQ,
        line_noise_freq=LINE_NOISE_FREQ,
        highpass_freq=HIGHPASS_FREQ,
        lowpass_freq=LOWPASS_FREQ,
        ic_threshold=IC_REJECTION_THRESHOLD,
        epoch_duration=EPOCH_DURATION,
        epoch_overlap_frac=EPOCH_OVERLAP,
        amplitude_reject_uv=AMPLITUDE_REJECT_UV,
        random_state=RANDOM_STATE,
        use_cuda=MNE_CUDA
    )

    elapsed = time.time() - t0
    result['elapsed_s'] = elapsed
    active_attempts += 1
    results.append(result)

    if result['status'] == 'success':
        success_count += 1
        shape = result['epoch_shape']
        print(f"Done {elapsed:.1f}s | {shape[0]} epochs | {result['n_bad_channels']} bad ch | {result['n_rejected_ics']} ICs rejected")
    else:
        failed_count += 1
        print(f"FAILED: {result.get('error', 'unknown')}")

    current_success_rate = success_count / max(1, active_attempts)
    wandb_subject_rows.append({
        'subject': subject_id,
        'status': result['status'],
        'elapsed_s': round(elapsed, 3),
        'n_bad_channels': result.get('n_bad_channels', 0),
        'n_rejected_ics': result.get('n_rejected_ics', 0),
        'n_epochs_before': result.get('n_epochs_before', 0),
        'n_epochs_after': result.get('n_epochs_after', 0),
        'bad_segments_s': result.get('bad_segments_s', 0.0),
        'success_rate_running': current_success_rate,
    })

    if wandb_run is not None:
        wandb_run.log({
            'preprocessing/subject_index': idx + 1,
            'preprocessing/subject_total': len(subject_dirs),
            'preprocessing/elapsed_s': elapsed,
            'preprocessing/bad_channels': result.get('n_bad_channels', 0),
            'preprocessing/rejected_ics': result.get('n_rejected_ics', 0),
            'preprocessing/epochs_before': result.get('n_epochs_before', 0),
            'preprocessing/epochs_after': result.get('n_epochs_after', 0),
            'preprocessing/bad_segments_s': result.get('bad_segments_s', 0.0),
            'preprocessing/success_count': success_count,
            'preprocessing/failed_count': failed_count,
            'preprocessing/missing_count': missing_count,
            'preprocessing/skipped_existing_count': skipped_count,
            'preprocessing/active_attempts': active_attempts,
            'preprocessing/current_success_rate': current_success_rate,
        }, step=idx + 1)

    if (idx + 1) % 10 == 0:
        print(f"W&B progress checkpoint: {idx+1}/{len(subject_dirs)} subjects logged")

total_time = time.time() - start_time
print('\n' + '=' * 70)
print('BATCH COMPLETE')
print('=' * 70)
n_success = sum(1 for r in results if r.get('status') == 'success')
n_failed = sum(1 for r in results if r.get('status') == 'failed')
print(f'Total time: {total_time/60:.1f} minutes')
print(f'Successful: {n_success}/{len(results)}')
print(f'Failed: {n_failed}/{len(results)}')
print(f'Skipped existing: {skipped_count}')
print(f'Missing files: {missing_count}')
print(f'Active success rate: {success_count}/{max(1, active_attempts)} = {success_count / max(1, active_attempts):.1%}')

LEMON DATASET - FULL BATCH PREPROCESSING
Subjects to process: 220
Output directory: G:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\data\LEMON_preprocessed
Target sampling rate: 250 Hz
GPU acceleration: NO

[  1/220] sub-010002 - Processing... 

C:\Users\milon\AppData\Local\Temp\ipykernel_10796\748837628.py:86: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])


Done 547.5s | 508 epochs | 1 bad ch | 24 ICs rejected
[  2/220] sub-010003 - Processing... 

C:\Users\milon\AppData\Local\Temp\ipykernel_10796\748837628.py:86: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw_clean = mne.add_reference_channels(raw_clean, ref_channels=[ch])


Done 229.1s | 501 epochs | 1 bad ch | 7 ICs rejected
[  3/220] sub-010004 - Processing... Done 1230.2s | 501 epochs | 0 bad ch | 21 ICs rejected
[  4/220] sub-010005 - Processing... 

KeyboardInterrupt: 

In [ ]:
# Sub-step 6: Summary and CSV export
successful = [r for r in results if r.get('status') == 'success']

if successful:
    df = pd.DataFrame(successful)

    print('=' * 60)
    print(f'PREPROCESSING SUMMARY ({len(successful)} subjects)')
    print('=' * 60)

    print('\nBad channels per subject:')
    print(f"  Mean +/- SD: {df['n_bad_channels'].mean():.1f} +/- {df['n_bad_channels'].std():.1f}")

    print('\nRejected ICA components per subject:')
    print(f"  Mean +/- SD: {df['n_rejected_ics'].mean():.1f} +/- {df['n_rejected_ics'].std():.1f}")

    n_epochs = [r['epoch_shape'][0] for r in successful]
    print('\nClean epochs per subject:')
    print(f'  Mean +/- SD: {np.mean(n_epochs):.0f} +/- {np.std(n_epochs):.0f}')

    summary_path = os.path.join(OUTPUT_DIR, 'preprocessing_summary.csv')
    df.to_csv(summary_path, index=False)
    print(f'\nSummary saved to: {summary_path}')

    if wandb_run is not None:
        wandb_table = wandb.Table(dataframe=pd.DataFrame(wandb_subject_rows))
        wandb_run.log({
            'preprocessing/final_success_rate': success_count / max(1, active_attempts),
            'preprocessing/total_subjects': len(subject_dirs),
            'preprocessing/successful_subjects': success_count,
            'preprocessing/failed_subjects': failed_count,
            'preprocessing/missing_subjects': missing_count,
            'preprocessing/skipped_existing_subjects': skipped_count,
            'preprocessing/active_attempts': active_attempts,
            'preprocessing/summary_table': wandb_table,
        })
        wandb_run.summary['final_success_rate'] = success_count / max(1, active_attempts)
        wandb_run.summary['successful_subjects'] = success_count
        wandb_run.summary['failed_subjects'] = failed_count
        wandb_run.summary['missing_subjects'] = missing_count
        wandb_run.summary['skipped_existing_subjects'] = skipped_count
        wandb_run.summary['active_attempts'] = active_attempts
        wandb_run.summary['summary_csv'] = summary_path
        wandb_run.finish()
else:
    print('No successful results to summarize.')
    if wandb_run is not None:
        wandb_run.summary['final_success_rate'] = 0.0
        wandb_run.summary['successful_subjects'] = 0
        wandb_run.summary['failed_subjects'] = failed_count
        wandb_run.summary['missing_subjects'] = missing_count
        wandb_run.summary['skipped_existing_subjects'] = skipped_count
        wandb_run.summary['active_attempts'] = active_attempts
        wandb_run.finish()